# Übung 2.2: Ein kleines CNN für MNIST mit PyTorch

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/german_version/exercise_2_2_mnist_cnn_pytorch_de.ipynb)

Dieses Notebook führt ein kleines Convolutional Neural Network (CNN) mit PyTorch ein. Der Datensatz ist MNIST: 28 x 28 Graustufenbilder handgeschriebener Ziffern.

Sie üben:

- Laden eines Bilddatensatzes mit `torchvision`
- Inspizieren von Bild-Batches
- Aufbau eines kleinen CNN mit `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d` und `nn.Linear`
- Training mit `CrossEntropyLoss` und `Adam`
- Verfolgen von Trainings- und Validierungsleistung
- Evaluation mit Accuracy, Confusion Matrix und Beispielvorhersagen


## 0. Colab-Setup

Führen Sie dies zuerst in Colab aus. Der MNIST-Datensatz wird automatisch von `torchvision` heruntergeladen.


In [ ]:
!pip -q install torch torchvision torchinfo torchview graphviz scikit-learn pandas matplotlib seaborn

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchinfo import summary
from torchview import draw_graph
from IPython.display import display

from sklearn.metrics import ConfusionMatrixDisplay, classification_report

sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. MNIST laden

MNIST enthält handgeschriebene Ziffern von 0 bis 9. Jedes Bild hat einen Kanal und eine Größe von 28 x 28 Pixeln.

Für eine schnellere Übung verwenden wir das vollständige Testset, aber nur einen Teil des Trainingssets. Sie können `TRAIN_SUBSET_SIZE` erhöhen, wenn Sie bessere Accuracy möchten.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

DATA_DIR = Path("data")
full_train = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

TRAIN_SUBSET_SIZE = 12000
VAL_SIZE = 2000
TRAIN_SIZE = TRAIN_SUBSET_SIZE - VAL_SIZE

small_train, _ = random_split(
    full_train,
    [TRAIN_SUBSET_SIZE, len(full_train) - TRAIN_SUBSET_SIZE],
    generator=torch.Generator().manual_seed(SEED),
)
train_dataset, val_dataset = random_split(
    small_train,
    [TRAIN_SIZE, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print("Train / val / test:", len(train_dataset), len(val_dataset), len(test_dataset))
xb, yb = next(iter(train_loader))
print("Image batch:", xb.shape)
print("Label batch:", yb.shape)


## 2. Einen Bild-Batch inspizieren

Die Tensorform ist `[batch_size, channels, height, width]`. Für MNIST ist `channels` gleich `1`, weil die Bilder Graustufenbilder sind.


In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
axes = axes.ravel()
for ax, image, label in zip(axes, images[:18], labels[:18]):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(f"label: {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### Aufgabe 1: Bilder betrachten

Betrachten Sie den Batch und beantworten Sie kurz:

- Welche Ziffern sehen ähnlich aus?
- Welche Beispiele könnten sogar für Menschen schwierig sein?
- Warum unterscheidet sich ein Bildmodell vom tabellarischen Modell in Übung 2.1?


## 3. Ein kleines CNN definieren

Ein CNN lernt kleine Filter, die über das Bild gleiten. Frühe Filter erkennen oft einfache Muster wie Kanten und Ecken. Spätere Schichten kombinieren diese Muster zu ziffernspezifischer Evidenz.

Dieses Modell ist bewusst klein, damit es in Colab schnell trainiert.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(DEVICE)
print(model)


In [ ]:
with torch.no_grad():
    sample_logits = model(images[:4].to(DEVICE))

print("Input shape: ", images[:4].shape)
print("Output shape:", sample_logits.shape)
print("One row of logits:", sample_logits[0].cpu().numpy().round(3))


## 4. CNN-Struktur visualisieren

Vor dem Training inspizieren wir die Modellstruktur. Die Summary-Tabelle zeigt, wie sich die Tensorform nach jeder Schicht verändert und wie viele trainierbare Parameter jede Schicht hat.

Die Graphansicht zeigt dasselbe Modell als Fluss vom Eingabebild zu den Klassenlogits.


In [ ]:
summary(
    model,
    input_size=(1, 1, 28, 28),
    col_names=["input_size", "output_size", "num_params", "kernel_size"],
    device=DEVICE.type,
)


In [ ]:
model_graph = draw_graph(
    model,
    input_size=(1, 1, 28, 28),
    device=DEVICE.type,
    graph_name="SmallCNN",
    depth=3,
    expand_nested=True,
)

display(model_graph.visual_graph)


## 5. CNN trainieren

Die Trainingsschritte sind dieselben wie in Übung 2.1:

1. Logits mit einem Forward Pass berechnen
2. Loss berechnen
3. `loss.backward()` ausführen
4. Parameter mit `optimizer.step()` aktualisieren

Der Unterschied liegt in der Architektur: CNN-Schichten sind für Bilder ausgelegt.


In [ ]:
def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss += loss.item() * len(images)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(images)

    return total_loss / total, correct / total

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 8
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item() * len(images)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += len(images)

    train_loss = train_loss_total / train_total
    train_acc = train_correct / train_total
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
        }
    )

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
        f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
    )

history_df = pd.DataFrame(history)


## 6. Lernkurven plotten

Die Validierungskurve hilft zu erkennen, ob das Modell noch besser wird oder anfängt zu overfitten.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=history_df, x="epoch", y="train_loss", marker="o", label="train", ax=axes[0])
sns.lineplot(data=history_df, x="epoch", y="val_loss", marker="o", label="validation", ax=axes[0])
axes[0].set_title("Loss")
axes[0].set_ylabel("Cross-entropy loss")

sns.lineplot(data=history_df, x="epoch", y="train_accuracy", marker="o", label="train", ax=axes[1])
sns.lineplot(data=history_df, x="epoch", y="val_accuracy", marker="o", label="validation", ax=axes[1])
axes[1].set_title("Accuracy")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.show()


### Aufgabe 2: Trainingseinstellungen ändern

Ändern Sie eine Einstellung und trainieren Sie erneut:

- `EPOCHS`: testen Sie `2`, `8` und `12`
- `TRAIN_SUBSET_SIZE`: testen Sie `3000` und `12000`
- erste Conv-Kanäle: ändern Sie `16` zu `8` oder `32`
- Learning Rate: ändern Sie `0.001` zu `0.0003` oder `0.003`

Was verbessert die Geschwindigkeit? Was verbessert die Accuracy?


## 7. Auf dem Testset evaluieren

Das Testset sollte erst am Ende verwendet werden, nachdem Sie die Einstellungen gewählt haben.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss:     {test_loss:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(DEVICE))
        predictions = logits.argmax(dim=1).cpu()
        all_predictions.append(predictions)
        all_labels.append(labels)

y_pred = torch.cat(all_predictions).numpy()
y_true = torch.cat(all_labels).numpy()

print(classification_report(y_true, y_pred, digits=3))

fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("MNIST CNN confusion matrix")
plt.tight_layout()
plt.show()


## 8. Korrekte und falsche Vorhersagen inspizieren

Die nützlichsten Fehler sind oft visuell: Sie zeigen, wo das Modell ähnlich aussehende Ziffern verwechselt.


In [ ]:
test_images, test_labels = next(iter(DataLoader(test_dataset, batch_size=256, shuffle=True)))
model.eval()
with torch.no_grad():
    logits = model(test_images.to(DEVICE)).cpu()
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

wrong = torch.where(preds != test_labels)[0]
right = torch.where(preds == test_labels)[0]

fig, axes = plt.subplots(2, 6, figsize=(11, 4.5))
for ax, idx in zip(axes[0], right[:6]):
    ax.imshow(test_images[idx].squeeze(0), cmap="gray")
    ax.set_title(f"true {test_labels[idx].item()} / pred {preds[idx].item()}")
    ax.axis("off")

for ax, idx in zip(axes[1], wrong[:6]):
    ax.imshow(test_images[idx].squeeze(0), cmap="gray")
    ax.set_title(f"true {test_labels[idx].item()} / pred {preds[idx].item()}")
    ax.axis("off")

axes[0, 0].set_ylabel("correct")
axes[1, 0].set_ylabel("wrong")
plt.tight_layout()
plt.show()


## 9. Kurze Reflexion

Beantworten Sie kurz:

- Was macht `Conv2d`, was eine tabellarische lineare Schicht nicht macht?
- Warum verwenden wir `MaxPool2d` in diesem kleinen CNN?
- Welche Ziffern werden am häufigsten verwechselt?
- Was würden Sie als Nächstes versuchen, um das Modell zu verbessern?

Ziel ist ein funktionierendes mentales Modell des CNN-Trainings, nicht ein perfekter MNIST-Score.
